**MGMT298D: Science and Strategy of AI**
# Week 5: Transfer Learning

#### We build a handbags-vs-shoes classifier two ways: first training a small CNN entirely from scratch, then freezing a pretrained VGG16 backbone and training only a tiny classification head on top. Live webcam demos before and after show the difference.

---
# 1 · Setup & Data

#### Download the dataset from GitHub and split into train / val / test. Images are resized to 224×224 to match what VGG16 expects.

In [ ]:
import os, pathlib, requests
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

keras.utils.set_random_seed(42)

In [ ]:
API      = 'https://api.github.com/repos/ucla-anderson-SSAI/SSAI/contents/handbags-shoes'
base_dir = pathlib.Path('handbags-shoes')

for category in ('handbags', 'shoes'):
    files = sorted(requests.get(f'{API}/{category}').json(), key=lambda f: f['name'])
    for split, sl in [('train', slice(0,50)), ('validation', slice(50,75)), ('test', slice(75,None))]:
        dst = base_dir / split / category
        os.makedirs(dst, exist_ok=True)
        for f in files[sl]:
            out = dst / f['name']
            if not out.exists():
                out.write_bytes(requests.get(f['download_url']).content)

train_ds = keras.utils.image_dataset_from_directory(base_dir/'train',      image_size=(224,224), batch_size=32, label_mode='binary')
val_ds   = keras.utils.image_dataset_from_directory(base_dir/'validation', image_size=(224,224), batch_size=32, label_mode='binary')
test_ds  = keras.utils.image_dataset_from_directory(base_dir/'test',       image_size=(224,224), batch_size=32, label_mode='binary')

# Quick look at a sample of training images
plt.figure(figsize=(8, 3))
for imgs, labels in train_ds.take(1):
    for i in range(6):
        plt.subplot(1, 6, i+1)
        plt.imshow(imgs[i].numpy().astype('uint8'))
        plt.title('shoe' if labels[i]==1 else 'handbag', fontsize=9)
        plt.axis('off')
plt.tight_layout(); plt.show()

---
# 2 · Webcam Helper

#### Sets up `live_detect()` — a rolling webcam loop that grabs frames, classifies each one, and updates the display in place. We'll reuse this before and after transfer learning.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import time, io, json
from PIL import Image as PILImage

def live_detect(model_fn, title='Live Detection', n_frames=60, interval=0.1):
    """Stream webcam frames into a model and display a live confidence chart.
    model_fn : callable(arr) -> float probability of 'shoe'
    n_frames : how many frames to capture before stopping
    interval : seconds between frames (tune to your GPU/CPU speed)
    """
    # Build a single HTML panel: video on the left, bar chart on the right
    js_setup = Javascript('''
        window._stream = null;
        window._video  = null;

        async function startCam() {
            if (window._stream) return 'already running';

            // Outer flex container
            const wrap = document.createElement('div');
            wrap.id = 'live-wrap';
            wrap.style.cssText = 'display:inline-flex;align-items:center;gap:16px;padding:12px;background:#1a1a2e;border-radius:10px;';

            // Left: video
            const video = document.createElement('video');
            video.style.cssText = 'width:400px;border-radius:6px;flex-shrink:0;';

            // Right: chart panel
            const chartWrap = document.createElement('div');
            chartWrap.style.cssText = 'display:flex;flex-direction:column;justify-content:center;min-width:220px;';
            const chartTitle = document.createElement('div');
            chartTitle.id = 'chart-title';
            chartTitle.style.cssText = 'color:#fff;font-family:monospace;font-size:13px;margin-bottom:10px;text-align:center;';
            chartTitle.textContent = 'Waiting...';
            const svg = document.createElementNS('http://www.w3.org/2000/svg','svg');
            svg.id = 'chart-svg';
            svg.setAttribute('width','220'); svg.setAttribute('height','80');
            chartWrap.appendChild(chartTitle);
            chartWrap.appendChild(svg);

            wrap.appendChild(video);
            wrap.appendChild(chartWrap);
            document.body.appendChild(wrap);

            window._video  = video;
            window._stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = window._stream;
            await video.play();
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
            return 'started';
        }

        async function grabFrame() {
            const v = window._video;
            const c = document.createElement('canvas');
            c.width = v.videoWidth; c.height = v.videoHeight;
            c.getContext('2d').drawImage(v, 0, 0);
            return c.toDataURL('image/jpeg', 0.7);
        }

        function updateChart(handbagProb, shoeProb, topClass, title) {
            const svg    = document.getElementById('chart-svg');
            const titleEl = document.getElementById('chart-title');
            if (\!svg || \!titleEl) return;

            const conf = topClass === 'handbag' ? handbagProb : shoeProb;
            titleEl.innerHTML = title + \'<br><b style="color:#f1c40f;font-size:15px">&rarr; \' + topClass.toUpperCase() + \'</b> (\'  + (conf*100).toFixed(1) + \'%)\';

            const W = 220, barH = 22, gap = 14, padL = 68, padR = 10;
            const maxW = W - padL - padR;
            const labels = [\'handbag\', \'shoe\'];
            const probs  = [handbagProb, shoeProb];
            svg.innerHTML = \'\';

            labels.forEach((lbl, i) => {
                const y     = i * (barH + gap) + 4;
                const bw    = probs[i] * maxW;
                const color = lbl === topClass ? \'#e74c3c\' : \'#2c3e50\';

                // Label
                const text = document.createElementNS(\'http://www.w3.org/2000/svg\',\'text\');
                text.setAttribute(\'x\', padL - 6); text.setAttribute(\'y\', y + barH*0.72);
                text.setAttribute(\'text-anchor\',\'end\'); text.setAttribute(\'fill\',\'#ccc\');
                text.setAttribute(\'font-size\',\'12\'); text.setAttribute(\'font-family\',\'monospace\');
                text.textContent = lbl;
                svg.appendChild(text);

                // Bar background
                const bg = document.createElementNS(\'http://www.w3.org/2000/svg\',\'rect\');
                bg.setAttribute(\'x\', padL); bg.setAttribute(\'y\', y);
                bg.setAttribute(\'width\', maxW); bg.setAttribute(\'height\', barH);
                bg.setAttribute(\'fill\',\'#0d0d1a\'); bg.setAttribute(\'rx\',\'4\');
                svg.appendChild(bg);

                // Bar fill
                const rect = document.createElementNS(\'http://www.w3.org/2000/svg\',\'rect\');
                rect.setAttribute(\'x\', padL); rect.setAttribute(\'y\', y);
                rect.setAttribute(\'width\', bw); rect.setAttribute(\'height\', barH);
                rect.setAttribute(\'fill\', color); rect.setAttribute(\'rx\',\'4\');
                svg.appendChild(rect);

                // Percentage label
                const pct = document.createElementNS(\'http://www.w3.org/2000/svg\',\'text\');
                pct.setAttribute(\'x\', padL + Math.max(bw - 4, 30)); pct.setAttribute(\'y\', y + barH*0.72);
                pct.setAttribute(\'text-anchor\',\'end\'); pct.setAttribute(\'fill\',\'white\');
                pct.setAttribute(\'font-size\',\'11\'); pct.setAttribute(\'font-weight\',\'bold\');
                pct.setAttribute(\'font-family\',\'monospace\');
                pct.textContent = (probs[i]*100).toFixed(1) + \'%\';
                svg.appendChild(pct);
            });
        }

        function stopCam() {
            if (window._stream) { window._stream.getTracks().forEach(t => t.stop()); window._stream = null; }
        }
    ''')
    display(js_setup)
    eval_js('startCam()')

    for i in range(n_frames):
        data      = eval_js('grabFrame()')
        img_bytes = b64decode(data.split(',')[1])
        img       = PILImage.open(io.BytesIO(img_bytes)).resize((224, 224))
        arr       = np.expand_dims(np.array(img), axis=0).astype('float32')

        p_shoe = model_fn(arr)
        probs  = {'handbag': 1 - p_shoe, 'shoe': p_shoe}
        top    = max(probs, key=probs.get)

        # Update the SVG chart and title label directly in the browser
        js_call = f'updateChart({probs["handbag"]:.4f}, {probs["shoe"]:.4f}, {json.dumps(top)}, {json.dumps(title)})'
        eval_js(js_call)
        time.sleep(interval)

    eval_js('stopCam()')
    print('Done.')

---
# 3 · Baseline: CNN Trained from Scratch

#### A small CNN with randomly initialised weights, trained on just 100 images. With so little data and no prior knowledge, it tends to struggle — this sets the bar we want to beat.

#### The model has ~25.7M parameters, almost all of them in the first Dense layer. Every single one is randomly initialised and learned from scratch on our tiny dataset.

In [ ]:
baseline_cnn = models.Sequential([
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='baseline_cnn')

baseline_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
baseline_cnn.fit(train_ds, epochs=10, validation_data=val_ds)

acc_baseline = baseline_cnn.evaluate(test_ds)[1]
print(f'Baseline CNN test accuracy: {acc_baseline:.4f}')

---
# 4 · Live Detection — Before Transfer Learning

#### Point your webcam at a handbag or shoe and capture. This is the baseline CNN making a cold guess with no pretrained knowledge.

In [ ]:
def baseline_predict(arr):
    return float(baseline_cnn.predict(arr, verbose=0)[0][0])

try:
    live_detect(baseline_predict, title='Before Transfer Learning (Baseline CNN)')
except Exception as err:
    print(err)

---
# 5 · Transfer Learning

#### We freeze VGG16's ImageNet-trained convolutional base and use it as a fixed feature extractor. Every image is passed through once to produce 7×7×512 feature maps, then we train a small dense head on top of those features. The backbone never changes — only the head learns.

#### VGG16's frozen backbone has ~14.7M parameters — none of them update during our training. The head we add on top has only ~26K trainable parameters. That's roughly 600× fewer than training from scratch, which is why it learns so effectively from just 100 images.

In [ ]:
# Load VGG16 without its classifier top; freeze all weights
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
vgg_base.trainable = False

# Extract features once and cache — makes head training very fast
def extract_features(ds):
    feats, labs = [], []
    for imgs, y in ds:
        feats.append(vgg_base.predict(preprocess_input(imgs), verbose=0))
        labs.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labs)

train_feats, train_labels = extract_features(train_ds)
val_feats,   val_labels   = extract_features(val_ds)
test_feats,  test_labels  = extract_features(test_ds)
print(f'Feature shape: {train_feats.shape}  (samples × 7 × 7 × 512)')

In [ ]:
# Small classification head trained on top of the frozen features
tl_head = models.Sequential([
    layers.Flatten(input_shape=(7, 7, 512)),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='transfer_learning_head')

tl_head.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
tl_head.fit(train_feats, train_labels, epochs=10, batch_size=32,
            validation_data=(val_feats, val_labels))

acc_tl = tl_head.evaluate(test_feats, test_labels)[1]
print(f'Transfer learning test accuracy: {acc_tl:.4f}')

# Quick comparison
plt.bar(['Baseline CNN', 'Transfer Learning'], [acc_baseline, acc_tl], color=['#c0392b', '#2980b9'])
plt.ylabel('Test Accuracy'); plt.ylim(0.4, 1.02)
plt.title('Baseline vs Transfer Learning')
plt.show()

---
# 6 · Live Detection — After Transfer Learning

#### Same webcam demo, now using the transfer learning model. Compare its confidence against what you saw in Section 4.

In [ ]:
def tl_predict(arr):
    feats = vgg_base.predict(preprocess_input(arr.copy()), verbose=0)
    return float(tl_head.predict(feats, verbose=0)[0][0])

try:
    live_detect(tl_predict, title='After Transfer Learning (VGG16 backbone)')
except Exception as err:
    print(err)